In [ ]:
import sys
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import torch

# Navigate to repo root regardless of where Colab/Jupyter started CWD
if not os.path.isdir("src"):
    for _candidate in ["..", "/content/urban-change-detection"]:
        if os.path.isdir(os.path.join(_candidate, "src")):
            os.chdir(_candidate)
            break

sys.path.insert(0, "src")
DATA_ROOT = "data/levir"

from data.dataset import make_dataloaders
from torchgeo.datasets import LEVIRCDPlus

In [ ]:
# --- Chargement du dataset brut pour l'exploration ---
base_train = LEVIRCDPlus(root=DATA_ROOT, split="train", download=False)
base_test  = LEVIRCDPlus(root=DATA_ROOT, split="test",  download=False)

print(f"Images train : {len(base_train)}")
print(f"Images test  : {len(base_test)}")

In [4]:
# --- Structure d'un sample ---
sample = base_train[0]
t1   = sample["image"][0]
t2   = sample["image"][1]
mask = sample["mask"]

print(f"T1 shape   : {t1.shape}     dtype : {t1.dtype}")
print(f"T2 shape   : {t2.shape}     dtype : {t2.dtype}")
print(f"Mask shape : {mask.shape}   dtype : {mask.dtype}")
print(f"Mask values: {mask.unique().tolist()}")

T1 shape   : torch.Size([3, 1024, 1024])     dtype : torch.float32
T2 shape   : torch.Size([3, 1024, 1024])     dtype : torch.float32
Mask shape : torch.Size([1, 1024, 1024])   dtype : torch.int64
Mask values: [0, 1]


In [ ]:
# --- Taux de déséquilibre de classe ---

change_rates = []
for i in range(len(base_train)):
    m = base_train[i]["mask"]
    rate = (m == 1).sum().item() / m.numel() * 100
    change_rates.append(rate)

print("Taux de pixels 'changé', split train")
print(f"  Moyenne : {np.mean(change_rates):.2f}%")
print(f"  Médiane : {np.median(change_rates):.2f}%")
print(f"  Min     : {np.min(change_rates):.2f}%")
print(f"  Max     : {np.max(change_rates):.2f}%")
print()
print(f"  → Prédire 'rien ne change' partout donne {100 - np.mean(change_rates):.1f}% d'accuracy.")
print(f"    C'est pourquoi on utilise F1 et IoU sur la classe 'changé' uniquement.")

In [ ]:
# --- Visualisation de 4 paires ---
def to_rgb(tensor):
    # Normalisation par percentile pour éviter que les valeurs extrêmes
    # n'écrasent le contraste de l'affichage.
    img = tensor.numpy().transpose(1, 2, 0).astype(np.float32)
    for c in range(3):
        p2, p98 = np.percentile(img[:, :, c], 2), np.percentile(img[:, :, c], 98)
        img[:, :, c] = np.clip((img[:, :, c] - p2) / (p98 - p2 + 1e-6), 0, 1)
    return img

fig, axes = plt.subplots(4, 3, figsize=(14, 18))
fig.suptitle("LEVIR-CD, Exemples de paires bi-temporelles", fontsize=14, y=1.01)

for row in range(4):
    s    = base_train[row]
    t1   = s["image"][0]
    t2   = s["image"][1]
    mask = s["mask"][0].numpy()

    rgb1    = to_rgb(t1)
    rgb2    = to_rgb(t2)
    overlay = rgb2.copy()
    overlay[mask == 1] = [1.0, 0.0, 0.0]

    axes[row, 0].imshow(rgb1);    axes[row, 0].set_title(f"Paire {row+1}, T1 (avant)");      axes[row, 0].axis("off")
    axes[row, 1].imshow(rgb2);    axes[row, 1].set_title(f"Paire {row+1}, T2 (après)");      axes[row, 1].axis("off")
    axes[row, 2].imshow(overlay); axes[row, 2].set_title(f"Paire {row+1}, changements (rouge)"); axes[row, 2].axis("off")

plt.tight_layout()
plt.savefig("eda_paires.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
# --- Visualisation de patches 256×256 ---
# Les images 1024×1024 sont trop grandes pour le GPU.
# On les découpe en patches de 256×256 (16 par image).

train_loader, val_loader, test_loader = make_dataloaders(
    root=DATA_ROOT, patch_size=256, batch_size=8
)

print(f"Patches train : {len(train_loader.dataset)}")
print(f"Patches val   : {len(val_loader.dataset)}")
print(f"Patches test  : {len(test_loader.dataset)}")

batch = next(iter(train_loader))
print(f"\nShape T1   : {batch['t1'].shape}")
print(f"Shape T2   : {batch['t2'].shape}")
print(f"Shape mask : {batch['mask'].shape}")

In [ ]:
# --- Visualisation de 4 patches du premier batch ---
fig, axes = plt.subplots(4, 3, figsize=(10, 14))
fig.suptitle("LEVIR-CD, Patches 256×256 après découpage", fontsize=13, y=1.01)

for row in range(4):
    t1   = batch["t1"][row]
    t2   = batch["t2"][row]
    mask = batch["mask"][row, 0].numpy()

    rgb1    = to_rgb(t1)
    rgb2    = to_rgb(t2)
    overlay = rgb2.copy()
    overlay[mask == 1] = [1.0, 0.0, 0.0]

    axes[row, 0].imshow(rgb1);    axes[row, 0].set_title(f"Patch {row+1}, T1"); axes[row, 0].axis("off")
    axes[row, 1].imshow(rgb2);    axes[row, 1].set_title(f"Patch {row+1}, T2"); axes[row, 1].axis("off")
    axes[row, 2].imshow(overlay); axes[row, 2].set_title(f"Patch {row+1}, mask"); axes[row, 2].axis("off")

plt.tight_layout()
plt.savefig("eda_patches.png", dpi=100, bbox_inches="tight")
plt.show()

In [ ]:
# --- Résumé EDA ---
print("=" * 50)
print("Résumé, LEVIR-CD")
print("=" * 50)
print(f"  Images originales  : {len(base_train)} train / {len(base_test)} test")
print(f"  Résolution         : 1024×1024 px @ 0.5 m/px")
print(f"  Canaux             : RGB (3 bandes)")
print(f"  Patches 256×256    : {len(train_loader.dataset)} train / {len(test_loader.dataset)} test")
print(f"  Déséquilibre       : {np.mean(change_rates):.1f}% changé / {100-np.mean(change_rates):.1f}% stable")
print(f"  Métrique principale: F1 + IoU sur classe 'changé'")
print(f"  Loss prévue        : BCE pondérée + Dice")

In [ ]:
# --- Cellule : évaluation de la baseline CVA ---
from models.baseline import change_vector_analysis, otsu_threshold, compute_metrics

all_preds  = []
all_masks  = []

print("Évaluation CVA sur le test set...")
for i, batch in enumerate(test_loader):
    t1   = batch["t1"]
    t2   = batch["t2"]
    mask = batch["mask"]

    magnitude = change_vector_analysis(t1, t2)
    pred      = otsu_threshold(magnitude)

    all_preds.append(pred)
    all_masks.append(mask)

    if i % 50 == 0:
        print(f"  Batch {i+1}/{len(test_loader)}")

preds  = torch.cat(all_preds,  dim=0)
masks  = torch.cat(all_masks,  dim=0)
metrics = compute_metrics(preds, masks)

print()
print("=" * 40)
print("Baseline CVA, résultats sur test set")
print("=" * 40)
print(f"  F1        : {metrics['f1']:.4f}")
print(f"  IoU       : {metrics['iou']:.4f}")
print(f"  Précision : {metrics['precision']:.4f}")
print(f"  Rappel    : {metrics['recall']:.4f}")